# Legacy CDW Data Ingestion Pipeline

This notebook ingests data from the legacy Corporate Data Warehouse (CDW) tables,
applies type conversions, validates data quality, and writes clean Delta Lake tables.

## Source Tables
| Legacy Table | Description | Known Issues |
|---|---|---|
| `CDW_BORR_MSTR` | Borrower master records | Credit scores as strings, dates as MM/DD/YYYY |
| `CDW_LN_PROD` | Loan product catalog | Amounts with commas |
| `CDW_LN_ACCT` | Loan accounts (denormalized) | SSN last-4 contains phone digits, no FK enforcement |
| `CDW_PMT_HIST` | Payment history | Component sum ≠ total for some records |

## Target: Delta Lake tables with proper types, validated data, and quality flags.

## Step 1: Configuration & Spark Session Setup

Initialize the Spark session with Delta Lake support. Define source/target paths
and quality threshold parameters.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DecimalType,
    IntegerType, DateType, BooleanType, TimestampType
)
from datetime import datetime

# Spark session (in Databricks this is pre-configured as `spark`)
spark = SparkSession.builder \
    .appName("LegacyCDW_Ingestion") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Configuration
LEGACY_SCHEMA = "legacy_cdw"        # Source schema/database
TARGET_SCHEMA = "modern_loans"       # Target Delta Lake schema
QUALITY_LOG_PATH = "/mnt/data/quality_logs/"  # Path for quality issue logs

# Data quality thresholds
CREDIT_SCORE_MIN = 300
CREDIT_SCORE_MAX = 850
VALID_LOAN_STATUSES = ["ACT", "CLO", "DFT", "FRB"]
VALID_PAYMENT_STATUSES = ["PST", "REV", "NSF", "PND"]
VALID_PAYMENT_TYPES = ["REG", "EXT", "PRT", "PRE"]
VALID_PROPERTY_TYPES = ["SFR", "CND", "MFR", "TWN"]

print(f"Ingestion started at {datetime.now()}")

## Step 2: Define Reusable Transformation UDFs

These functions handle the core type conversions from legacy VARCHAR fields:

1. **Amount parsing** — Strips `$`, commas, spaces; handles accounting negatives `(1,200.00)`
2. **Date parsing** — Converts `MM/DD/YYYY` strings to proper `DateType`
3. **Status expansion** — Maps short codes (ACT, CLO) to readable values (ACTIVE, CLOSED)

All transformations include error handling that produces NULL instead of crashing
the pipeline on bad data.

In [ ]:
from pyspark.sql.functions import udf, col, when, lit, regexp_replace, trim
from decimal import Decimal, InvalidOperation


# --- Amount Parsing ---
# Legacy amounts are VARCHAR with commas: '285,000', '1,487.02', '$525,000'
# Some may have dollar signs, spaces, or accounting-format negatives: '(1,200.00)'
def parse_legacy_amount_expr(column):
    """
    PySpark expression to parse legacy amount strings to DecimalType.
    Handles: commas, dollar signs, spaces, accounting negatives.
    Returns NULL for unparseable values (instead of crashing the pipeline).
    """
    # Strip whitespace, dollar signs, and commas
    cleaned = trim(column)
    cleaned = regexp_replace(cleaned, r'[\$,\s]', '')
    # Handle accounting-format negatives: (1200.00) → -1200.00
    cleaned = when(
        cleaned.startswith('(') & cleaned.endswith(')'),
        F.concat(lit('-'), F.expr(f"substring({cleaned._jc.toString()}, 2, length({cleaned._jc.toString()}) - 2)"))
    ).otherwise(cleaned)
    # Cast to decimal — returns NULL on failure
    return cleaned.cast(DecimalType(12, 2))


def safe_parse_amount(column):
    """
    Simplified amount parser using regexp_replace for common patterns.
    Strips $, commas, spaces then casts to Decimal(12,2).
    Unparseable values become NULL (not an error).
    """
    cleaned = regexp_replace(trim(col(column)), r'[\$,\s]', '')
    # Handle accounting negatives: (value) → -value
    cleaned = when(
        cleaned.rlike(r'^\(.*\)$'),
        F.concat(lit('-'), regexp_replace(cleaned, r'[\(\)]', ''))
    ).otherwise(cleaned)
    return cleaned.cast(DecimalType(12, 2))


# --- Date Parsing ---
# Legacy dates are VARCHAR(10) in MM/DD/YYYY format
# Invalid dates (13/32/2020, 00/00/0000) should become NULL, not crash
def safe_parse_date(column):
    """
    Parses MM/dd/yyyy string to DateType.
    Returns NULL for invalid/malformed dates.
    """
    return F.to_date(col(column), 'MM/dd/yyyy')


# --- Status Code Expansion ---
# Short cryptic codes → readable values for the modern schema
LOAN_STATUS_MAP = {"ACT": "ACTIVE", "CLO": "CLOSED", "DFT": "DEFAULT", "FRB": "FORBEARANCE"}
PAYMENT_STATUS_MAP = {"PST": "POSTED", "REV": "REVERSED", "NSF": "NSF", "PND": "PENDING"}
PAYMENT_TYPE_MAP = {"REG": "REGULAR", "EXT": "EXTRA", "PRT": "PARTIAL", "PRE": "PREPAYMENT"}
PROPERTY_TYPE_MAP = {"SFR": "Single Family", "CND": "Condominium", "MFR": "Multi-Family", "TWN": "Townhouse"}


def expand_status(column, mapping):
    """
    Maps short legacy codes to expanded modern values.
    Unrecognized codes are preserved as-is (flagged in quality checks).
    """
    expr = col(column)
    for code, expanded in mapping.items():
        expr = when(col(column) == code, lit(expanded)).otherwise(expr)
    return expr


print("Transformation functions defined.")

## Step 3: Ingest & Transform Borrower Master (`CDW_BORR_MSTR`)

**Transformations applied:**
- `BORR_DOB_DT` (VARCHAR `MM/DD/YYYY`) → `date_of_birth` (DATE)
- `BORR_CRDT_SCR` (VARCHAR `'745'`) → `credit_score` (INTEGER, validated 300-850)
- `BORR_ANN_INCM` (VARCHAR `'92,500'`) → `annual_income` (DECIMAL, commas stripped)
- `BORR_CRET_DT` / `BORR_UPDT_DT` → `created_at` / `updated_at` (TIMESTAMP)
- `BORR_STAT_CD` (`ACT`) → `status` (`ACTIVE`)
- `BORR_REC_TYP` → dropped (not needed in modern schema)

**Quality checks:**
- Credit score outside FICO range (300-850) → flagged, set to NULL
- Unparseable dates → NULL with quality flag
- Unparseable income → NULL with quality flag

In [ ]:
# Read legacy borrower table
df_borrowers_raw = spark.table(f"{LEGACY_SCHEMA}.CDW_BORR_MSTR")

print(f"Raw borrower records: {df_borrowers_raw.count()}")
df_borrowers_raw.printSchema()

# Apply transformations
df_borrowers = df_borrowers_raw.select(
    # Direct copy fields
    col("BORR_ID").alias("external_id"),
    col("BORR_FST_NM").alias("first_name"),
    col("BORR_LST_NM").alias("last_name"),
    col("BORR_MID_INIT").alias("middle_initial"),
    col("BORR_SSN_ENCR").alias("ssn_hash"),
    
    # Date conversion: MM/DD/YYYY string → DATE type
    safe_parse_date("BORR_DOB_DT").alias("date_of_birth"),
    
    # Address fields (direct copy)
    col("BORR_ADDR_LN1").alias("address_line1"),
    col("BORR_ADDR_LN2").alias("address_line2"),
    col("BORR_CTY_NM").alias("city"),
    col("BORR_ST_CD").alias("state"),
    col("BORR_ZIP_CD").alias("zip_code"),
    col("BORR_PH_NBR").alias("phone"),
    col("BORR_EMAIL_ADDR").alias("email"),
    
    # Credit score: string → integer with FICO range validation
    # Scores outside 300-850 are set to NULL (flagged below)
    when(
        col("BORR_CRDT_SCR").cast(IntegerType()).between(CREDIT_SCORE_MIN, CREDIT_SCORE_MAX),
        col("BORR_CRDT_SCR").cast(IntegerType())
    ).otherwise(lit(None).cast(IntegerType())).alias("credit_score"),
    
    col("BORR_EMP_STAT").alias("employment_status"),
    
    # Annual income: strip commas, parse to decimal
    safe_parse_amount("BORR_ANN_INCM").alias("annual_income"),
    
    # Timestamps from date strings
    safe_parse_date("BORR_CRET_DT").cast(TimestampType()).alias("created_at"),
    safe_parse_date("BORR_UPDT_DT").cast(TimestampType()).alias("updated_at"),
    
    # Status expansion: ACT → ACTIVE, INA → INACTIVE
    expand_status("BORR_STAT_CD", {"ACT": "ACTIVE", "INA": "INACTIVE"}).alias("status"),
    
    # Quality flags for downstream consumers
    when(
        col("BORR_CRDT_SCR").cast(IntegerType()).isNull() |
        ~col("BORR_CRDT_SCR").cast(IntegerType()).between(CREDIT_SCORE_MIN, CREDIT_SCORE_MAX),
        lit(True)
    ).otherwise(lit(False)).alias("_dq_invalid_credit_score"),
    
    when(
        safe_parse_date("BORR_DOB_DT").isNull() & col("BORR_DOB_DT").isNotNull(),
        lit(True)
    ).otherwise(lit(False)).alias("_dq_invalid_dob")
)

print(f"Transformed borrower records: {df_borrowers.count()}")
df_borrowers.show(truncate=False)

## Step 4: Ingest & Transform Loan Products (`CDW_LN_PROD`)

**Transformations applied:**
- `PROD_TERM_MOS` (VARCHAR `'360'`) → `term_months` (INTEGER)
- `PROD_MIN_AMT` / `PROD_MAX_AMT` (VARCHAR `'50,000'`) → DECIMAL, commas stripped
- `PROD_STAT_CD` (`ACT`) → `is_active` (BOOLEAN: ACT=true, else=false)
- `PROD_EFF_DT` / `PROD_EXP_DT` → DATE type

Loan products are a small reference table (~5 records) so quality issues here
have outsized impact since they affect ALL loans referencing that product.

In [ ]:
# Read legacy loan products table
df_products_raw = spark.table(f"{LEGACY_SCHEMA}.CDW_LN_PROD")

# Apply transformations
df_products = df_products_raw.select(
    col("PROD_CD").alias("code"),
    col("PROD_DESC_TXT").alias("name"),
    col("PROD_TYP_CD").alias("type"),
    
    # Term months: string → integer
    col("PROD_TERM_MOS").cast(IntegerType()).alias("term_months"),
    
    col("PROD_RT_TYP").alias("rate_type"),
    
    # Amount fields: strip commas, parse to decimal
    safe_parse_amount("PROD_MIN_AMT").alias("min_amount"),
    safe_parse_amount("PROD_MAX_AMT").alias("max_amount"),
    
    # Status to boolean: ACT → true, anything else → false
    when(col("PROD_STAT_CD") == "ACT", lit(True))
        .otherwise(lit(False)).alias("is_active"),
    
    # Dates: MM/DD/YYYY string → DATE
    safe_parse_date("PROD_EFF_DT").alias("effective_date"),
    safe_parse_date("PROD_EXP_DT").alias("expiration_date")
)

print(f"Loan products: {df_products.count()}")
df_products.show(truncate=False)

## Step 5: Ingest & Transform Loan Accounts (`CDW_LN_ACCT`)

This is the most complex transformation due to denormalization and multiple field types.

**Transformations applied:**
- `LN_ORIG_AMT`, `LN_CURR_BAL`, `LN_PMT_AMT`, `LN_ESCROW_BAL`, `PROP_APRS_VAL` → DECIMAL (commas stripped)
- `LN_INT_RT`, `LN_LTV_PCT` → DECIMAL (no commas, direct parse)
- `LN_TERM_MOS`, `LN_DLQ_DAYS` → INTEGER
- All date fields → DATE type
- `LN_STAT_CD` → expanded status string
- `PROP_TYP_CD` → expanded property type
- Denormalized borrower fields (`BORR_FST_NM`, `BORR_LST_NM`, `BORR_SSN_LST4`) → **dropped**

**Critical anomaly handled:**
- `BORR_SSN_LST4` is known to contain phone number last-4 digits, NOT actual SSN.
  This field is intentionally excluded from the modern schema.

**FK validation:**
- `BORR_ID` validated against borrower master
- `PROD_CD` validated against loan products

In [ ]:
# Read legacy loan accounts table
df_loans_raw = spark.table(f"{LEGACY_SCHEMA}.CDW_LN_ACCT")

print(f"Raw loan account records: {df_loans_raw.count()}")

# Apply transformations — drop denormalized borrower fields (use FK instead)
df_loans = df_loans_raw.select(
    col("LN_ACCT_NBR").alias("account_number"),
    col("BORR_ID").alias("borrower_external_id"),  # FK reference to borrowers
    # NOTE: BORR_FST_NM, BORR_LST_NM, BORR_SSN_LST4 intentionally DROPPED
    # BORR_SSN_LST4 is CORRUPT — contains phone number last 4, not SSN (see DATA_ANOMALY_REPORT.md)
    col("PROD_CD").alias("product_code"),           # FK reference to products
    
    # Monetary amounts: strip commas, parse to decimal
    safe_parse_amount("LN_ORIG_AMT").alias("original_amount"),
    safe_parse_amount("LN_CURR_BAL").alias("current_balance"),
    
    # Interest rate: direct decimal parse (no commas)
    col("LN_INT_RT").cast(DecimalType(5, 3)).alias("interest_rate"),
    
    # Term & payment
    col("LN_TERM_MOS").cast(IntegerType()).alias("term_months"),
    safe_parse_amount("LN_PMT_AMT").alias("monthly_payment"),
    
    # Date fields: MM/DD/YYYY → DATE
    safe_parse_date("LN_ORIG_DT").alias("origination_date"),
    safe_parse_date("LN_MAT_DT").alias("maturity_date"),
    safe_parse_date("LN_1ST_PMT_DT").alias("first_payment_date"),
    safe_parse_date("LN_NXT_PMT_DT").alias("next_payment_date"),
    
    # Status expansion: ACT → ACTIVE, CLO → CLOSED, DFT → DEFAULT, FRB → FORBEARANCE
    expand_status("LN_STAT_CD", LOAN_STATUS_MAP).alias("status"),
    
    # Delinquency & escrow
    col("LN_DLQ_DAYS").cast(IntegerType()).alias("delinquency_days"),
    safe_parse_amount("LN_ESCROW_BAL").alias("escrow_balance"),
    col("LN_LTV_PCT").cast(DecimalType(5, 2)).alias("ltv_percent"),
    
    # Property fields
    col("PROP_ADDR_LN1").alias("property_address"),
    col("PROP_CTY_NM").alias("property_city"),
    col("PROP_ST_CD").alias("property_state"),
    col("PROP_ZIP_CD").alias("property_zip"),
    expand_status("PROP_TYP_CD", PROPERTY_TYPE_MAP).alias("property_type"),
    safe_parse_amount("PROP_APRS_VAL").alias("appraised_value"),
    
    # Audit timestamps
    safe_parse_date("LN_CRET_DT").cast(TimestampType()).alias("created_at"),
    safe_parse_date("LN_UPDT_DT").cast(TimestampType()).alias("updated_at"),
    
    # Quality flags
    when(
        ~col("LN_STAT_CD").isin(VALID_LOAN_STATUSES),
        lit(True)
    ).otherwise(lit(False)).alias("_dq_invalid_status")
)

# FK validation: flag orphaned references
valid_borrower_ids = df_borrowers.select("external_id")
valid_product_codes = df_products.select("code")

df_loans = df_loans \
    .join(
        valid_borrower_ids,
        df_loans.borrower_external_id == valid_borrower_ids.external_id,
        "left"
    ) \
    .withColumn("_dq_orphaned_borrower",
        when(valid_borrower_ids.external_id.isNull(), lit(True)).otherwise(lit(False))
    ) \
    .drop(valid_borrower_ids.external_id)

df_loans = df_loans \
    .join(
        valid_product_codes,
        df_loans.product_code == valid_product_codes.code,
        "left"
    ) \
    .withColumn("_dq_orphaned_product",
        when(valid_product_codes.code.isNull(), lit(True)).otherwise(lit(False))
    ) \
    .drop(valid_product_codes.code)

print(f"Transformed loan records: {df_loans.count()}")
df_loans.show(truncate=False)

## Step 6: Ingest & Transform Payment History (`CDW_PMT_HIST`)

**Transformations applied:**
- All amount fields (`PMT_AMT`, `PMT_PRIN_AMT`, `PMT_INT_AMT`, `PMT_ESCROW_AMT`, `PMT_LATE_FEE`) → DECIMAL
- All date fields → DATE type
- `PMT_TYP_CD` → expanded type (REGULAR, EXTRA, PARTIAL, PREPAYMENT)
- `PMT_STAT_CD` → expanded status (POSTED, REVERSED, NSF, PENDING)

**Critical anomaly check — Payment Component Validation:**

The invariant `total = principal + interest + escrow + late_fee` is validated.
Known violator: Loan `LN-2019-00142` has payments where components sum to
$1,887.02 but total is stated as $1,487.02 (consistent +$400 delta).
These records are flagged but not rejected — downstream consumers decide how to handle.

In [ ]:
# Read legacy payment history table
df_payments_raw = spark.table(f"{LEGACY_SCHEMA}.CDW_PMT_HIST")

print(f"Raw payment records: {df_payments_raw.count()}")

# Apply transformations
df_payments = df_payments_raw.select(
    col("PMT_SEQ_NBR").alias("legacy_payment_id"),
    col("LN_ACCT_NBR").alias("loan_account_number"),  # FK to loan_accounts
    
    # Dates: MM/DD/YYYY → DATE
    safe_parse_date("PMT_DT").alias("payment_date"),
    
    # Monetary amounts: strip commas, parse to decimal
    safe_parse_amount("PMT_AMT").alias("total_amount"),
    safe_parse_amount("PMT_PRIN_AMT").alias("principal_amount"),
    safe_parse_amount("PMT_INT_AMT").alias("interest_amount"),
    safe_parse_amount("PMT_ESCROW_AMT").alias("escrow_amount"),
    safe_parse_amount("PMT_LATE_FEE").alias("late_fee"),
    
    # Type & status expansion
    expand_status("PMT_TYP_CD", PAYMENT_TYPE_MAP).alias("type"),
    expand_status("PMT_STAT_CD", PAYMENT_STATUS_MAP).alias("status"),
    
    # Additional dates
    safe_parse_date("PMT_RECV_DT").alias("received_date"),
    safe_parse_date("PMT_PROC_DT").alias("processed_date"),
    safe_parse_date("PMT_CRET_DT").cast(TimestampType()).alias("created_at"),
    safe_parse_date("PMT_UPDT_DT").cast(TimestampType()).alias("updated_at"),
    
    # Quality flags
    when(
        ~col("PMT_STAT_CD").isin(VALID_PAYMENT_STATUSES),
        lit(True)
    ).otherwise(lit(False)).alias("_dq_invalid_status"),
    when(
        ~col("PMT_TYP_CD").isin(VALID_PAYMENT_TYPES),
        lit(True)
    ).otherwise(lit(False)).alias("_dq_invalid_type")
)

# --- CRITICAL: Payment Component Validation ---
# Verify: total_amount == principal + interest + escrow + late_fee
# This catches Anomaly #1 (payment mismatch for LN-2019-00142)
df_payments = df_payments.withColumn(
    "_computed_total",
    col("principal_amount") + col("interest_amount") + col("escrow_amount") + col("late_fee")
).withColumn(
    "_dq_component_mismatch",
    when(
        F.abs(col("_computed_total") - col("total_amount")) > 0.01,
        lit(True)
    ).otherwise(lit(False))
).withColumn(
    "_dq_mismatch_delta",
    when(
        col("_dq_component_mismatch") == True,
        col("_computed_total") - col("total_amount")
    ).otherwise(lit(None).cast(DecimalType(12, 2)))
)

# FK validation: flag payments referencing non-existent loans
valid_loan_accounts = df_loans.select(col("account_number").alias("_valid_acct"))
df_payments = df_payments \
    .join(valid_loan_accounts,
          df_payments.loan_account_number == valid_loan_accounts._valid_acct,
          "left") \
    .withColumn("_dq_orphaned_loan",
        when(col("_valid_acct").isNull(), lit(True)).otherwise(lit(False))
    ) \
    .drop("_valid_acct")

# Show flagged records
print("\n=== PAYMENT COMPONENT MISMATCHES (Anomaly #1) ===")
df_payments.filter(col("_dq_component_mismatch") == True) \
    .select("legacy_payment_id", "loan_account_number", "total_amount",
            "_computed_total", "_dq_mismatch_delta") \
    .show(truncate=False)

print(f"Total payment records: {df_payments.count()}")
print(f"Records with component mismatch: {df_payments.filter(col('_dq_component_mismatch')).count()}")

## Step 7: Data Quality Summary Report

Aggregate all quality flags across all tables to produce a summary.
This report can be used to:
1. Track data quality improvement over time
2. Decide whether to proceed with migration or fix source data first
3. Alert downstream consumers about unreliable fields

In [ ]:
# Data Quality Summary
print("=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

# Borrower quality
borrower_count = df_borrowers.count()
invalid_credit = df_borrowers.filter(col("_dq_invalid_credit_score")).count()
invalid_dob = df_borrowers.filter(col("_dq_invalid_dob")).count()
print(f"\n[BORROWERS] Total: {borrower_count}")
print(f"  - Invalid credit scores: {invalid_credit} ({100*invalid_credit/max(borrower_count,1):.1f}%)")
print(f"  - Invalid date of birth: {invalid_dob} ({100*invalid_dob/max(borrower_count,1):.1f}%)")

# Loan quality
loan_count = df_loans.count()
orphaned_borrowers = df_loans.filter(col("_dq_orphaned_borrower")).count()
orphaned_products = df_loans.filter(col("_dq_orphaned_product")).count()
invalid_loan_status = df_loans.filter(col("_dq_invalid_status")).count()
print(f"\n[LOANS] Total: {loan_count}")
print(f"  - Orphaned borrower refs: {orphaned_borrowers} ({100*orphaned_borrowers/max(loan_count,1):.1f}%)")
print(f"  - Orphaned product refs:  {orphaned_products} ({100*orphaned_products/max(loan_count,1):.1f}%)")
print(f"  - Invalid status codes:   {invalid_loan_status} ({100*invalid_loan_status/max(loan_count,1):.1f}%)")

# Payment quality
payment_count = df_payments.count()
component_mismatches = df_payments.filter(col("_dq_component_mismatch")).count()
orphaned_loans = df_payments.filter(col("_dq_orphaned_loan")).count()
invalid_pmt_status = df_payments.filter(col("_dq_invalid_status")).count()
invalid_pmt_type = df_payments.filter(col("_dq_invalid_type")).count()
print(f"\n[PAYMENTS] Total: {payment_count}")
print(f"  - Component mismatches:   {component_mismatches} ({100*component_mismatches/max(payment_count,1):.1f}%)")
print(f"  - Orphaned loan refs:     {orphaned_loans} ({100*orphaned_loans/max(payment_count,1):.1f}%)")
print(f"  - Invalid status codes:   {invalid_pmt_status} ({100*invalid_pmt_status/max(payment_count,1):.1f}%)")
print(f"  - Invalid type codes:     {invalid_pmt_type} ({100*invalid_pmt_type/max(payment_count,1):.1f}%)")

print("\n" + "=" * 70)

## Step 8: Write to Delta Lake

Write the validated, transformed DataFrames as Delta Lake tables.

**Delta Lake schema design decisions:**
- All monetary fields use `DECIMAL(12,2)` for exact arithmetic (no floating point)
- Dates are proper `DATE` type for efficient partition pruning and range queries
- Quality flag columns (`_dq_*`) are included for transparency — consumers can
  filter on these to exclude questionable records
- `_computed_total` column in payments allows consumers to choose which total to trust
- Tables are partitioned by status (borrowers/loans) for common query patterns
- Payments partitioned by year-month of `payment_date` for time-series queries

**Merge strategy:** OVERWRITE for initial load; switch to MERGE for incremental.

In [ ]:
# Write borrowers to Delta Lake
df_borrowers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("status") \
    .saveAsTable(f"{TARGET_SCHEMA}.borrowers")

print(f"✓ Written {df_borrowers.count()} borrowers to {TARGET_SCHEMA}.borrowers")

# Write loan products to Delta Lake (small table, no partitioning)
df_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{TARGET_SCHEMA}.loan_products")

print(f"✓ Written {df_products.count()} products to {TARGET_SCHEMA}.loan_products")

# Write loan accounts to Delta Lake
df_loans.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("status") \
    .saveAsTable(f"{TARGET_SCHEMA}.loan_accounts")

print(f"✓ Written {df_loans.count()} loans to {TARGET_SCHEMA}.loan_accounts")

# Write payments to Delta Lake (partitioned by payment month for time-series queries)
df_payments_partitioned = df_payments.withColumn(
    "payment_year_month",
    F.date_format(col("payment_date"), "yyyy-MM")
)

df_payments_partitioned.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("payment_year_month") \
    .saveAsTable(f"{TARGET_SCHEMA}.payments")

print(f"✓ Written {df_payments_partitioned.count()} payments to {TARGET_SCHEMA}.payments")

## Step 9: Write Data Quality Issues to Log Table

Persist all flagged records to a quality log Delta table for auditing.
This allows data stewards to:
1. Query all records with quality issues across all source tables
2. Track resolution over time
3. Generate compliance reports for data governance

In [ ]:
# Collect quality issues into a unified log table
from pyspark.sql import Row

quality_issues = []

# Borrower issues
borrower_issues = df_borrowers.filter(
    col("_dq_invalid_credit_score") | col("_dq_invalid_dob")
).select(
    lit("CDW_BORR_MSTR").alias("source_table"),
    col("external_id").alias("record_id"),
    when(col("_dq_invalid_credit_score"), lit("INVALID_CREDIT_SCORE"))
        .when(col("_dq_invalid_dob"), lit("INVALID_DATE_OF_BIRTH"))
        .alias("issue_type"),
    lit("MEDIUM").alias("severity"),
    F.current_timestamp().alias("detected_at")
)

# Loan issues
loan_issues = df_loans.filter(
    col("_dq_orphaned_borrower") | col("_dq_orphaned_product") | col("_dq_invalid_status")
).select(
    lit("CDW_LN_ACCT").alias("source_table"),
    col("account_number").alias("record_id"),
    when(col("_dq_orphaned_borrower"), lit("ORPHANED_BORROWER_REF"))
        .when(col("_dq_orphaned_product"), lit("ORPHANED_PRODUCT_REF"))
        .when(col("_dq_invalid_status"), lit("INVALID_STATUS_CODE"))
        .alias("issue_type"),
    when(col("_dq_orphaned_borrower") | col("_dq_orphaned_product"), lit("HIGH"))
        .otherwise(lit("MEDIUM")).alias("severity"),
    F.current_timestamp().alias("detected_at")
)

# Payment issues (including the critical component mismatch)
payment_issues = df_payments.filter(
    col("_dq_component_mismatch") | col("_dq_orphaned_loan") |
    col("_dq_invalid_status") | col("_dq_invalid_type")
).select(
    lit("CDW_PMT_HIST").alias("source_table"),
    col("legacy_payment_id").alias("record_id"),
    when(col("_dq_component_mismatch"), lit("PAYMENT_COMPONENT_MISMATCH"))
        .when(col("_dq_orphaned_loan"), lit("ORPHANED_LOAN_REF"))
        .when(col("_dq_invalid_status"), lit("INVALID_STATUS_CODE"))
        .when(col("_dq_invalid_type"), lit("INVALID_TYPE_CODE"))
        .alias("issue_type"),
    when(col("_dq_component_mismatch"), lit("CRITICAL"))
        .when(col("_dq_orphaned_loan"), lit("HIGH"))
        .otherwise(lit("MEDIUM")).alias("severity"),
    F.current_timestamp().alias("detected_at")
)

# Union all issues and write to quality log
df_quality_log = borrower_issues.unionByName(loan_issues).unionByName(payment_issues)

df_quality_log.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(f"{TARGET_SCHEMA}.data_quality_log")

print(f"\n✓ Logged {df_quality_log.count()} quality issues to {TARGET_SCHEMA}.data_quality_log")
df_quality_log.groupBy("source_table", "issue_type", "severity").count().orderBy("severity").show(truncate=False)

## Step 10: Verification & Schema Inspection

Final verification that Delta Lake tables were written correctly with proper types.
This confirms the schema design is sound and all type conversions landed correctly.

In [ ]:
# Verify Delta Lake table schemas
print("=" * 70)
print("DELTA LAKE SCHEMA VERIFICATION")
print("=" * 70)

for table in ["borrowers", "loan_products", "loan_accounts", "payments", "data_quality_log"]:
    print(f"\n--- {TARGET_SCHEMA}.{table} ---")
    df = spark.table(f"{TARGET_SCHEMA}.{table}")
    df.printSchema()
    print(f"Row count: {df.count()}")

# Verify type conversions worked (spot check)
print("\n" + "=" * 70)
print("TYPE CONVERSION SPOT CHECKS")
print("=" * 70)

# Check borrower date_of_birth is a real DATE
sample = spark.table(f"{TARGET_SCHEMA}.borrowers").select(
    "external_id", "date_of_birth", "credit_score", "annual_income"
).limit(3)
print("\nBorrower type conversions:")
sample.show(truncate=False)
print(f"date_of_birth type: {dict(sample.dtypes)['date_of_birth']}")
print(f"credit_score type:  {dict(sample.dtypes)['credit_score']}")
print(f"annual_income type: {dict(sample.dtypes)['annual_income']}")

# Check payment amounts are decimal
sample_pmt = spark.table(f"{TARGET_SCHEMA}.payments").select(
    "legacy_payment_id", "total_amount", "payment_date", "_dq_component_mismatch"
).limit(3)
print("\nPayment type conversions:")
sample_pmt.show(truncate=False)
print(f"total_amount type: {dict(sample_pmt.dtypes)['total_amount']}")
print(f"payment_date type: {dict(sample_pmt.dtypes)['payment_date']}")

print("\n✓ All Delta Lake tables verified. Ingestion pipeline complete.")